# Week 09 — Python Solution Lab
## Rotational Dynamics

**Companion to `notebooks/Week_09.ipynb`.** This notebook contains *fully worked Python
solutions* to selected problems from that week's problem set — one at **each difficulty level**.

Every solution follows the course's core workflow:

> **Diagram → Principle → Equation → Predict → Verify**

The markdown cell states the problem, identifies the governing principle, and gives the **hand
prediction you should make before running anything**. The code cell then computes the result and
*verifies* it — typically by a second independent method (energy vs. forces, symbolic vs.
numerical, closed form vs. simulation) — and includes `assert` checks against the known answer.

### How to use this notebook

1. **Attempt the problem in `Week_09.ipynb` first.** These solutions are worth very little
   if you read them before trying.
2. Make the hand prediction. Write it down.
3. Run the code cell and compare.
4. **Change a number and re-run.** Every solution is written so that the parameters sit at the
   top; the sweeps and plots update automatically. Ask "what if the mass doubled?" and answer it
   in ten seconds.

### Why the code looks like this

These are not minimal answer-generators. Each one demonstrates something Python does that hand
algebra cannot: parameter sweeps, root-finding, numerical integration, symbolic differentiation,
or a cross-check to machine precision. The physics is the point; the code is how we prove the
physics is right.

---


### Solutions in this notebook

| Level | Problem | Topic | Python technique |
|---|---|---|---|
| **L1 · Basic** | `P3` | Torque, Angular Acceleration, and a Disk | linear↔rotational analogy table |
| **L2 · Intermediate** | `P7` | Solid vs Hollow Sphere Rolling Down a Ramp | shape-factor ranking, the rolling race |
| **L3 · Challenge** | `P9` | Atwood Machine with a Massive Pulley | 3×3 coupled FBDs, ideal-pulley limit |

---

## L1 · Basic — P3: Torque, Angular Acceleration, and a Disk

> **Problem (Week_09.ipynb, L1 — P3).** A constant torque of $8.0$ N·m is applied to a solid
> disk of mass $4.0$ kg and radius $0.20$ m, initially at rest. Find the angular acceleration and
> the angular velocity after $3.0$ s.

**Diagram → Principle.** $\tau = I\alpha$ is the rotational $F = ma$. Then rotational kinematics,
exactly parallel to the linear case.

**Equation.** $I = \tfrac12MR^2$, $\alpha = \tau/I$, $\omega = \alpha t$.

**Hand prediction.** $I = 0.080$ kg·m², $\alpha = 100$ rad/s², $\omega = 300$ rad/s.

**What Python adds.** We put the **linear–rotational analogy** on screen as a side-by-side table
($m \leftrightarrow I$, $F \leftrightarrow \tau$, $v \leftrightarrow \omega$), because that
mapping is the whole point of the week. We then verify the answer a second way through the
work–energy theorem for rotation, $\tau\theta = \tfrac12I\omega^2$.

In [ ]:
# ═══ W09 · L1 · P3 — tau = I alpha, and the linear/rotational dictionary ═══
import numpy as np

# --- MODEL --------------------------------------------------------------
tau, M, R, t = 8.0, 4.0, 0.20, 3.0

# --- PREDICT ------------------------------------------------------------
I     = 0.5 * M * R**2                 # solid disk about its central axis
alpha = tau / I
omega = alpha * t
theta = 0.5 * alpha * t**2

print(f"I     = (1/2) M R^2 = {I:.4f} kg*m^2")
print(f"alpha = tau / I     = {alpha:.2f} rad/s^2")
print(f"omega = alpha * t   = {omega:.2f} rad/s   after {t:.1f} s")
print(f"theta = {theta:.2f} rad = {theta/(2*np.pi):.2f} revolutions")

# --- VERIFY via the rotational work-energy theorem ----------------------
W  = tau * theta
KE = 0.5 * I * omega**2
print(f"\nwork done by the torque  W = tau*theta = {W:.2f} J")
print(f"rotational kinetic energy  = I w^2 / 2 = {KE:.2f} J   -> agrees")
assert np.isclose(W, KE)

# --- The dictionary that makes this week easy ---------------------------
# NOTE: this maps EQUATION FORMS, not numbers for the same object. A spinning
# disk has no single translational speed, so pairing its mass with a rim speed
# would be meaningless -- see the warning below the table.
print("\n  LINEAR QUANTITY        ROTATIONAL TWIN        this problem")
print("  " + "-"*62)
rows = [("position    x",        "angle        theta",  f"{theta:.2f} rad"),
        ("velocity    v",        "ang. velocity omega", f"{omega:.2f} rad/s"),
        ("accel       a",        "ang. accel   alpha",  f"{alpha:.2f} rad/s^2"),
        ("mass        m",        "inertia      I",      f"{I:.4f} kg*m^2"),
        ("force       F",        "torque       tau",    f"{tau:.2f} N*m"),
        ("F = m a",              "tau = I alpha",       f"{tau:.2f} = {I:.4f} x {alpha:.0f}"),
        ("KE = m v^2 / 2",       "KE = I omega^2 / 2",  f"{KE:.2f} J"),
        ("W = F d",              "W = tau theta",       f"{W:.2f} J")]
for a_, b_, c_ in rows:
    print(f"  {a_:22s} {b_:22s} {c_}")
print("\n  Read this as a dictionary between EQUATIONS: every rotational law has the")
print("  same form as its linear twin under x->theta, v->omega, a->alpha, m->I, F->tau.")
print("\n  CAUTION: it is NOT a conversion between numbers for one object. This disk")
print("  has no single translational speed -- different points move at different v.")
print(f"  Its rotational KE is {KE:.1f} J. Multiplying the whole mass by the RIM speed")
print(f"  would give (1/2) M v_rim^2 = {0.5*M*(omega*R)**2:.1f} J, which is 2x too big and")
print("  describes nothing physical. Only the equation FORMS correspond.")

# --- A rim point, to keep the connection concrete -----------------------
print(f"\nA point on the rim after {t:.0f} s:")
print(f"  tangential speed        v = omega R = {omega*R:.2f} m/s")
print(f"  tangential acceleration a = alpha R = {alpha*R:.2f} m/s^2")
print(f"  centripetal accel  a_c = omega^2 R = {omega**2*R:,.0f} m/s^2 "
      f"= {omega**2*R/9.81:,.0f} g   <- this is what tears rotors apart")

# --- CHECK --------------------------------------------------------------
assert abs(I - 0.08) < 1e-12 and abs(alpha - 100.0) < 1e-9 and abs(omega - 300.0) < 1e-9
print(f"\n[OK] Matches textbook answer: alpha = {alpha:.0f} rad/s^2, omega = {omega:.0f} rad/s")

## L2 · Intermediate — P7: Solid vs Hollow Sphere Rolling Down a Ramp

> **Problem (Week_09.ipynb, L2 — P7).** A solid sphere and a hollow sphere (both $M = 2.0$ kg,
> $R = 0.10$ m) roll without slipping down a $1.5$ m high ramp inclined at $30^\circ$. Find the
> speed of each at the bottom and the time difference between their arrivals.

**Diagram → Principle.** Rolling without slipping splits the released potential energy between
**translation and rotation**. The bigger the moment of inertia, the larger the share diverted to
spinning, and the slower the object arrives.

**Equation.** $Mgh = \tfrac12Mv^2\left(1 + \dfrac{I}{MR^2}\right)$, so
$v = \sqrt{\dfrac{2gh}{1+\beta}}$ with $\beta = I/MR^2$.

**Hand prediction.** $\beta = 2/5$ and $2/3$: $v_{\rm solid} = 4.585$ m/s,
$v_{\rm hollow} = 4.202$ m/s, $\Delta t = 0.119$ s.

> ⚠️ **Answer-key note.** The key prints $v_{\rm hollow} = 4.18$ m/s and $\Delta t = 0.10$ s.
> With $\beta = 2/3$ and $g = 9.81$ the exact values are $4.202$ m/s and $0.119$ s. This is
> **not** a rounding difference: $4.202$ rounds to $4.20$, not $4.18$, and $0.119$ rounds to
> $0.12$, not $0.10$. It is a small numerical inconsistency in the key — the method behind it is
> right, only the values drift. ($v_{\rm solid} = 4.58$ m/s is correct.)

**What Python adds.** The striking fact is that $M$ and $R$ **cancel entirely** — only the shape
factor $\beta$ survives. We demonstrate that by re-running with wildly different masses and radii,
then rank a whole family of shapes (sphere, disk, hoop, hollow sphere) in one table. That is the
"race down the ramp" demo, computed.

In [ ]:
# ═══ W09 · L2 · P7 — Rolling race: only the shape factor beta matters ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
M, R, h, theta_deg, g = 2.0, 0.10, 1.5, 30.0, 9.81
theta = np.radians(theta_deg)
d     = h / np.sin(theta)              # distance travelled along the ramp

def roll(beta, h=h, theta=theta, g=g):
    """Returns (v_bottom, a_along_ramp, t) for a body with I = beta M R^2."""
    v = np.sqrt(2*g*h / (1 + beta))
    a = g*np.sin(theta) / (1 + beta)
    t = v / a
    return v, a, t

beta_solid, beta_hollow = 2/5, 2/3
v_s, a_s, t_s = roll(beta_solid)
v_h, a_h, t_h = roll(beta_hollow)

print(f"ramp: h = {h} m at {theta_deg:.0f} deg  ->  length d = {d:.3f} m\n")
print(f"solid sphere  (beta = 2/5): v = {v_s:.3f} m/s, a = {a_s:.3f} m/s^2, t = {t_s:.3f} s")
print(f"hollow sphere (beta = 2/3): v = {v_h:.3f} m/s, a = {a_h:.3f} m/s^2, t = {t_h:.3f} s")
print(f"\ntime difference dt = {t_h - t_s:.3f} s   (the solid sphere wins)")

# --- VERIFY: distance check, and the energy split -----------------------
for nm, v, a, t, beta in (("solid ", v_s, a_s, t_s, beta_solid),
                          ("hollow", v_h, a_h, t_h, beta_hollow)):
    assert abs(0.5*a*t**2 - d) < 1e-9, "kinematics must reproduce the ramp length"
    KE_tr  = 0.5*M*v**2
    KE_rot = 0.5*(beta*M*R**2)*(v/R)**2
    print(f"  {nm}: KE_trans {KE_tr:6.3f} J + KE_rot {KE_rot:6.3f} J = {KE_tr+KE_rot:6.3f} J"
          f"   (M g h = {M*g*h:.3f} J)  rot share {100*KE_rot/(KE_tr+KE_rot):.1f}%")
    assert abs(KE_tr + KE_rot - M*g*h) < 1e-9

# --- VERIFY: mass and radius really do cancel ---------------------------
print("\n  changing M and R must not change the result:")
for Mt, Rt in ((0.02, 0.005), (2.0, 0.10), (200.0, 1.5)):
    vt = np.sqrt(2*g*h / (1 + beta_solid))          # no M, no R anywhere
    print(f"    M = {Mt:7.2f} kg, R = {Rt:5.3f} m -> v = {vt:.4f} m/s")
print("    -> a marble and a cannonball of the same shape arrive together.")

# --- The full shape ranking ---------------------------------------------
shapes = [("solid sphere",  2/5), ("solid disk/cylinder", 1/2),
          ("hollow sphere", 2/3), ("hoop / thin ring", 1.0),
          ("(sliding, frictionless)", 0.0)]
print(f"\n  {'shape':26s} {'beta':>6s} {'v (m/s)':>9s} {'t (s)':>8s}")
results = []
for nm, b in shapes:
    v, a, t = roll(b)
    results.append((nm, b, v, t))
    print(f"  {nm:26s} {b:6.3f} {v:9.3f} {t:8.3f}")
print("  -> the ranking depends ONLY on beta: mass concentrated near the axis wins.")

# --- Plot the race ------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.4, 4))
tmax = max(r[3] for r in results)
tt   = np.linspace(0, tmax, 400)
for (nm, b, v, t), col in zip(results,
        ["#2e7d32", "#1565c0", "#e65100", "crimson", "grey"]):
    a = g*np.sin(theta)/(1 + b)
    s = np.minimum(0.5*a*tt**2, d)
    ax.plot(tt, s, lw=2, color=col, label=f"{nm} ($\\beta$={b:.2f})")
ax.axhline(d, ls="--", c="k", lw=1, label=f"finish line, {d:.1f} m")
ax.set_xlabel("t (s)"); ax.set_ylabel("distance along ramp (m)")
ax.set_title("W09 P7 — the rolling race down a 30 deg ramp")
ax.grid(alpha=.3); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(v_s - 4.5849) < 5e-3, f"v_solid = {v_s}"
assert abs(v_h - 4.2021) < 5e-3, f"v_hollow = {v_h}"
assert abs((t_h - t_s) - 0.1190) < 5e-3, f"dt = {t_h - t_s}"
print(f"\n[OK] v_solid = {v_s:.3f} m/s, v_hollow = {v_h:.3f} m/s, dt = {t_h - t_s:.3f} s")
print(f"     The key prints 4.58 / 4.18 / 0.10. This is NOT rounding:")
print(f"       {v_h:.3f} rounds to {round(v_h, 2)}, not 4.18")
print(f"       {t_h-t_s:.3f} rounds to {round(t_h-t_s, 2)}, not 0.10")
print(f"     -> a small numerical inconsistency in the key, not a display artefact.")

## L3 · Challenge — P9: Atwood Machine with a Massive Pulley

> **Problem (Week_09.ipynb, L3 — P9).** An Atwood machine uses a solid disk pulley of
> $M = 3.0$ kg, $R = 0.08$ m. Masses $m_1 = 7.0$ kg and $m_2 = 5.0$ kg hang from an inextensible
> string that does not slip. Find (a) the acceleration, (b) $T_1$ and $T_2$, (c) the pulley's
> angular acceleration.

**Diagram → Principle.** The pulley now has inertia, so the two tensions are **no longer equal** —
their difference is exactly what angularly accelerates the pulley. Three FBDs, one constraint
$a = \alpha R$.

**Equation.** $m_1g - T_1 = m_1a$; $T_2 - m_2g = m_2a$; $(T_1 - T_2)R = I\alpha$, $I = \tfrac12MR^2$.

**Hand prediction.** $a = \dfrac{(m_1-m_2)g}{m_1+m_2+M/2} = \dfrac{19.62}{13.5} = 1.453$ m/s²;
$T_1 = m_1(g-a) = 58.50$ N, $T_2 = m_2(g+a) = 56.32$ N, $\alpha = a/R = 18.17$ rad/s².

> ⚠️ **The printed answer key for this problem is wrong.** It gives $a = 1.26$ m/s²,
> $T_1 = 59.85$ N, $T_2 = 55.44$ N, $\alpha = 15.75$ rad/s². Those values fail the pulley's own
> torque equation: $(T_1-T_2)R = 0.353$ N·m but $I\alpha = 0.151$ N·m — off by a factor of $2.3$.
> The denominator $m_1+m_2+M/2 = 13.5$ kg gives $a = 1.453$ m/s²; reproducing $1.26$ would need
> an effective pulley mass of $7.1$ kg, not $3.0$ kg. **The cell below verifies the corrected
> values against all three free-body equations and an independent energy check.**

**What Python adds.** We assemble the three coupled equations as a **matrix** and solve for
$(a, T_1, T_2)$ simultaneously — no substitution chain to get lost in. Then we sweep the pulley
mass from $0$ to $30$ kg and watch $T_1 \to T_2$ as $M \to 0$, recovering the massless-pulley
result as a limiting case. Seeing the ideal case fall out of the general one is the point.

In [ ]:
# ═══ W09 · L3 · P9 — Atwood with a massive pulley, solved as a 3x3 linear system ═══
import numpy as np
import matplotlib.pyplot as plt

g = 9.81
m1, m2, M, R = 7.0, 5.0, 3.0, 0.08

def atwood(m1, m2, M, R, g=9.81):
    """
    Unknowns u = [a, T1, T2], with a > 0 meaning m1 descends.
      (1) m1 g - T1 = m1 a        ->   m1 a + T1        = m1 g
      (2) T2 - m2 g = m2 a        ->   m2 a       - T2  = -m2 g
      (3) (T1 - T2) R = I a / R   ->   (I/R^2) a - T1 + T2 = 0
    """
    I = 0.5 * M * R**2
    A = np.array([[m1,      1.0,  0.0],
                  [m2,      0.0, -1.0],
                  [I/R**2, -1.0,  1.0]])
    b = np.array([m1*g, -m2*g, 0.0])
    return np.linalg.solve(A, b), I

(a, T1, T2), I = atwood(m1, m2, M, R)
alpha = a / R

print(f"pulley inertia I = (1/2) M R^2 = {I:.6f} kg*m^2\n")
print(f"(a) a     = {a:.4f} m/s^2")
print(f"(b) T1    = {T1:.4f} N   (heavier side, {m1:.0f} kg)")
print(f"    T2    = {T2:.4f} N   (lighter side, {m2:.0f} kg)")
print(f"    T1-T2 = {T1-T2:.4f} N   <- this difference is what spins the pulley")
print(f"(c) alpha = a / R = {alpha:.4f} rad/s^2")

# --- VERIFY 1: the closed-form shortcut ---------------------------------
a_closed = (m1 - m2)*g / (m1 + m2 + M/2)
print(f"\nclosed form a = (m1-m2)g / (m1+m2+M/2) = {a_closed:.4f} m/s^2 -> agrees")
assert np.isclose(a, a_closed)

# --- VERIFY 2: every individual FBD must balance ------------------------
assert np.isclose(m1*g - T1, m1*a),          "m1 FBD"
assert np.isclose(T2 - m2*g, m2*a),          "m2 FBD"
assert np.isclose((T1 - T2)*R, I*alpha),     "pulley torque equation"
print("all three free-body equations balance independently. [verified]")

# --- VERIFY 3: energy check after 1 m of travel -------------------------
d = 1.0
v = np.sqrt(2*a*d); w = v/R
lhs = (m1 - m2)*g*d
rhs = 0.5*m1*v**2 + 0.5*m2*v**2 + 0.5*I*w**2
print(f"\nafter {d:.0f} m: PE released {lhs:.4f} J = KE gained {rhs:.4f} J "
      f"(residual {abs(lhs-rhs):.1e})")
assert abs(lhs - rhs) < 1e-9

# --- SWEEP: recover the ideal massless pulley as a limit ----------------
Ms = np.linspace(0, 30, 400)
res = np.array([atwood(m1, m2, Mi, R)[0] for Mi in Ms])
a_s, T1_s, T2_s = res[:, 0], res[:, 1], res[:, 2]

print(f"\nlimit M -> 0:  a = {a_s[0]:.4f} m/s^2, T1 = {T1_s[0]:.4f} N, T2 = {T2_s[0]:.4f} N")
print(f"  ideal Atwood: a = (m1-m2)g/(m1+m2) = {(m1-m2)*g/(m1+m2):.4f}, "
      f"T = 2 m1 m2 g/(m1+m2) = {2*m1*m2*g/(m1+m2):.4f} N")
assert abs(T1_s[0] - T2_s[0]) < 1e-9, "with a massless pulley the tensions must be equal"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 3.9))
ax1.plot(Ms, a_s, color="#1565c0", lw=2.5)
ax1.axvline(M, ls=":", c="grey"); ax1.plot(M, a, "o", color="crimson", ms=9, zorder=5)
ax1.set_xlabel("pulley mass M (kg)"); ax1.set_ylabel("a (m/s$^2$)")
ax1.set_title("a heavier pulley slows everything"); ax1.grid(alpha=.3)

ax2.plot(Ms, T1_s, color="#2e7d32", lw=2, label="$T_1$")
ax2.plot(Ms, T2_s, color="#e65100", lw=2, label="$T_2$")
ax2.fill_between(Ms, T2_s, T1_s, color="grey", alpha=.2, label="$T_1-T_2$ spins the pulley")
ax2.axvline(M, ls=":", c="grey")
ax2.set_xlabel("pulley mass M (kg)"); ax2.set_ylabel("tension (N)")
ax2.set_title("the tensions split apart as M grows"); ax2.grid(alpha=.3); ax2.legend(fontsize=9)
plt.suptitle("W09 P9 — Atwood machine with a real pulley", y=1.03)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(a - 1.4533) < 1e-3, f"a = {a}"
assert abs(T1 - 58.4967) < 5e-3 and abs(T2 - 56.3167) < 5e-3
assert abs(alpha - 18.1667) < 1e-2

# --- Why the printed key cannot be right --------------------------------
aK, T1K, T2K, alK = 1.26, 59.85, 55.44, 15.75
print(f"\nANSWER-KEY AUDIT (the key prints a = {aK}, T1 = {T1K}, T2 = {T2K}, alpha = {alK})")
print(f"  pulley torque equation demands (T1-T2)R == I*alpha:")
print(f"    key:      {(T1K-T2K)*R:.5f}  vs  {I*alK:.5f}   -> MISMATCH by {(T1K-T2K)*R/(I*alK):.2f}x")
print(f"    ours:     {(T1-T2)*R:.5f}  vs  {I*alpha:.5f}   -> consistent")
print(f"  also a = (m1-m2)g/(m1+m2+M/2) = 19.62/{m1+m2+M/2:.1f} = {a:.4f}, not {aK}.")
print(f"  ({aK} would require an effective pulley mass of "
      f"{2*((m1-m2)*g/aK - m1 - m2):.1f} kg instead of {M:.1f} kg.)")
assert abs((T1K-T2K)*R - I*alK) > 0.1, "the key really is inconsistent"

print(f"\n[OK] CORRECTED answer: a = {a:.3f} m/s^2, T1 = {T1:.2f} N, "
      f"T2 = {T2:.2f} N, alpha = {alpha:.2f} rad/s^2")

---

## Self-check

**Every code cell above contains one or more `assert` checks.** If you run the whole notebook top
to bottom without an `AssertionError`, all of the numerical checks on this page have passed.
(The asserts sit just before each cell's closing summary, so the last thing you see is a printed
result — not the check itself.)

**Now transfer the skill.** Pick one unsolved problem from `Week_09.ipynb` at the level you
found hardest, and write the same five-part structure for it:

```python
# --- MODEL:   parameters at the top, with units in comments
# --- PREDICT: the closed-form answer
# --- VERIFY:  a SECOND, independent route to the same number
# --- CHECK:   assert against your hand prediction
```

The verify step is the one that matters. A result you have only computed one way is a result you
have not checked.
